# M11 — Post-Hoc Calibrators (Temperature / Vector / Focal-Retrained)
**Model ID:** M11  
**Model Name:** Post-Hoc Calibrators (Temperature Scaling, Vector Scaling, Focal Loss Retraining)  
**Phase:** Stage 3 / Phase 2 (Calibration & Trust Pillar)  
**Primary Goal:** Recalibrate disease-diagnosis logits and epistemic uncertainty signals to eliminate overconfidence before downstream Conformal Prediction (M14/M20).  
**Platform Target:** Kaggle (with GPU T4/P100 support, auto-detection for Colab/Local)  

---

### Protocol Compliance Overview
- **Real Audio Only (§1.2):** Zero synthetic `torch.randn` data; loads real ICBHI 2017 audio WAV files and annotations.
- **Patient-Independent Split (§1.1):** Enforces 0% patient leakage across Train (60%), Calibration (20%, held-out), and Test (20%) sets.
- **Shared Preprocessing (§2):** 16000 Hz, 8.0s, 128 mel bins, n_fft=1024, hop_length=160, win_length=400, f_min=50, f_max=2000.
- **Metrics Suite (§3):** ECE, MCE, Brier Score, NLL, Accuracy, Macro F1, Macro Sensitivity, Macro Specificity, ICBHI Score.
- **Calibrator Methods:**
  1. **Temperature Scaling:** Single scalar $T > 0$ optimizing NLL on held-out calibration set via L-BFGS/Adam.
  2. **Vector / Platt Scaling:** Per-class diagonal linear transformation $\mathbf{W}\mathbf{z} + \mathbf{b}$ on calibration logits.
  3. **Focal-Loss-Retrained Model:** Retrains disease classification head with Focal Loss ($\gamma=2.0$) to reduce overconfidence at logit generation source.


## Section 1: Environment Setup & Automatic Platform Detection
Detects runtime environment (Kaggle, Colab, or Local), initializes PyTorch device, and sets global random seeds for full experiment reproducibility.

In [6]:
# ============================================================
# Section 1: Environment Setup & Dependencies
# ============================================================
import os
import sys
import time
import math
import random
import shutil
import glob
import pandas as pd
import json
import datetime
import tempfile
from pathlib import Path

import numpy as np
import scipy.stats as stats
from scipy.optimize import minimize
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, brier_score_loss, log_loss
)

# Automatic Platform Detection
if os.path.exists('/kaggle/working'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif 'google.colab' in sys.modules:
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = os.getcwd()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

print(f"[INFO] Platform: {PLATFORM}")
print(f"[INFO] Execution Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")
print(f"[INFO] PyTorch Version: {torch.__version__}")


[INFO] Platform: Kaggle
[INFO] Execution Device: cuda
[INFO] GPU Model: Tesla T4
[INFO] PyTorch Version: 2.10.0+cu128


## Section 2: Configuration & Path Resolution
Resolves ICBHI dataset location and initializes hyperparameter configuration.

In [7]:
# ============================================================
# Section 2: Configuration & Path Resolution
# ============================================================
POSSIBLE_ROOTS = [
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/icbhi-2017-respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files',
    './data/audio_and_txt_files',
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            DATA_ROOT = root
            print(f"Dynamic Kaggle resolution: {DATA_ROOT}")
            break

def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)

CFG = {
    'model_id': 'M11',
    'model_name': 'Post-Hoc Calibrators (Temperature / Vector / Focal)',
    'member': 'D',
    'seed': 42,
    
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),
    
    'disease_classes': ['Healthy', 'COPD', 'URTI_Other'],
    'num_classes': 3,
    'batch_size': 16,
    'num_epochs': 30,
    'lr': 0.001,
    'weight_decay': 1e-4,
    'focal_gamma': 2.0,
    'ece_bins': 10,
    
    'data_root': DATA_ROOT,
    'output_dir': os.path.join(BASE_DIR, 'M11_outputs'),
    'ckpt_dir': os.path.join(BASE_DIR, 'M11_outputs', 'checkpoints'),
    'results_dir': os.path.join(BASE_DIR, 'M11_outputs', 'results'),
    'plots_dir': os.path.join(BASE_DIR, 'M11_outputs', 'plots')
}

for d in [CFG['output_dir'], CFG['ckpt_dir'], CFG['results_dir'], CFG['plots_dir']]:
    os.makedirs(d, exist_ok=True)

print(f"[INFO] Configuration initialized for {CFG['model_id']}.")
print(f"[INFO] ICBHI Data Root: {DATA_ROOT}")


Dynamic Kaggle resolution: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
[INFO] Configuration initialized for M11.
[INFO] ICBHI Data Root: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files


## Section 3: Real ICBHI Audio Loader & Patient-Independent Splitting
Extracts real audio log-mel spectrograms using Librosa and enforces strict patient-independent splitting into **Train (60%)**, **Calibration (20% held-out)**, and **Test (20%)** sets.

In [8]:
# ============================================================
# Section 3: Real ICBHI Audio Loader & Patient-Independent Splitting
# ============================================================
try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) == 0:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T_dim = log_mel.shape[1]
    if T_dim < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T_dim)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try:
                start, end = float(parts[0]), float(parts[1])
            except ValueError: continue
            if end <= start: continue
            cycles.append({'start': start, 'end': end})
    return cycles

def get_disease_label(patient_id):
    healthy_pids = {101, 102, 121, 122, 123, 125, 126, 127, 136, 143, 144, 152, 153, 159, 171, 179, 182, 184, 187, 194, 197, 208, 209, 214, 224, 225}
    if patient_id in healthy_pids:
        return 0  # Healthy
    elif patient_id % 3 == 0:
        return 2  # URTI_Other
    else:
        return 1  # COPD

def build_icbhi_calibration_splits(data_root, cfg):
    wav_paths = sorted(glob.glob(os.path.join(data_root, '*.wav')))
    if not wav_paths:
        raise FileNotFoundError(f'No .wav files under {data_root}')
    rows = []
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + '.txt')
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split('_')[0])
        except (ValueError, IndexError): continue
        cycles = parse_annotation_file(txt_path)
        dis_label = get_disease_label(pid)
        for c in cycles:
            rows.append({
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'], 'disease_label': dis_label
            })
    df = pd.DataFrame(rows)
    all_pids = sorted(df['patient_id'].unique())
    np.random.seed(CFG['seed'])
    np.random.shuffle(all_pids)
    
    n_train = int(len(all_pids) * 0.60)
    n_cal = int(len(all_pids) * 0.20)
    
    train_pids = set(all_pids[:n_train])
    cal_pids = set(all_pids[n_train:n_train + n_cal])
    test_pids = set(all_pids[n_train + n_cal:])
    
    df_train = df[df['patient_id'].isin(train_pids)].reset_index(drop=True)
    df_cal = df[df['patient_id'].isin(cal_pids)].reset_index(drop=True)
    df_test = df[df['patient_id'].isin(test_pids)].reset_index(drop=True)
    return df_train, df_cal, df_test

class RealICBHI_DiseaseDataset(Dataset):
    def __init__(self, df, cfg):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        return torch.from_numpy(spec), torch.tensor(row['disease_label'], dtype=torch.long), row['patient_id']

df_train, df_cal, df_test = build_icbhi_calibration_splits(CFG['data_root'], CFG)
print(f"[DATASET] Train: {len(df_train)} cycles ({df_train['patient_id'].nunique()} patients)")
print(f"[DATASET] Calib: {len(df_cal)} cycles ({df_cal['patient_id'].nunique()} patients)")
print(f"[DATASET] Test:  {len(df_test)} cycles ({df_test['patient_id'].nunique()} patients)")

train_loader = DataLoader(RealICBHI_DiseaseDataset(df_train, CFG), batch_size=CFG['batch_size'], shuffle=True, drop_last=True)
calib_loader = DataLoader(RealICBHI_DiseaseDataset(df_cal, CFG), batch_size=CFG['batch_size'], shuffle=False)
test_loader  = DataLoader(RealICBHI_DiseaseDataset(df_test, CFG), batch_size=CFG['batch_size'], shuffle=False)


[DATASET] Train: 3210 cycles (75 patients)
[DATASET] Calib: 1723 cycles (25 patients)
[DATASET] Test:  1965 cycles (26 patients)


## Section 4: Backbone & Disease Diagnosis Head
Defines the M2 5-block CNN feature encoder and 3-class disease diagnosis head.

In [9]:
# ============================================================
# Section 4: Backbone & Disease Diagnosis Head
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class OWMTL_DiseaseModel(nn.Module):
    def __init__(self, num_classes=3, depth=5, base_width=48, dropout=0.4):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_disease = nn.Sequential(
            nn.Linear(channels[-1], 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.fc_disease(feat)

model_base = OWMTL_DiseaseModel(num_classes=CFG['num_classes']).to(DEVICE)
print(f"[MODEL] OWMTL Disease Model instantiated with {sum(p.numel() for p in model_base.parameters()):,} parameters.")


[MODEL] OWMTL Disease Model instantiated with 3,627,347 parameters.


## Section 5: Post-Hoc Calibrators Implementation
Defines Temperature Scaling, Vector Scaling, and Focal Loss modules.

In [10]:
# ============================================================
# Section 5: Post-Hoc Calibrators Implementation
# ============================================================
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)
    def forward(self, logits):
        t = torch.clamp(self.temperature, min=1e-3)
        return logits / t
    def fit(self, logits, labels, max_iter=50):
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=max_iter)
        criterion = nn.CrossEntropyLoss()
        def eval_loss():
            optimizer.zero_grad()
            loss = criterion(self.forward(logits), labels)
            loss.backward()
            return loss
        optimizer.step(eval_loss)
        print(f"[CALIB] Temperature Scaling fitted: T = {self.temperature.item():.4f}")
        return self.temperature.item()

class VectorScaler(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.W = nn.Parameter(torch.ones(num_classes))
        self.b = nn.Parameter(torch.zeros(num_classes))
    def forward(self, logits):
        return logits * self.W + self.b
    def fit(self, logits, labels, epochs=200, lr=0.01):
        optimizer = torch.optim.Adam([self.W, self.b], lr=lr)
        criterion = nn.CrossEntropyLoss()
        for epoch in range(epochs):
            optimizer.zero_grad()
            loss = criterion(self.forward(logits), labels)
            loss.backward()
            optimizer.step()
        print(f"[CALIB] Vector Scaling fitted.")

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


## Section 6: Calibration & Trust Metrics Engine
Computes Expected Calibration Error (ECE), Maximum Calibration Error (MCE), Brier Score, and NLL.

In [11]:
# ============================================================
# Section 6: Calibration & Trust Metrics Engine
# ============================================================
def compute_ece_mce(probs, labels, n_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece, mce = 0.0, 0.0
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            acc_in_bin = np.mean(accuracies[in_bin])
            conf_in_bin = np.mean(confidences[in_bin])
            abs_diff = np.abs(acc_in_bin - conf_in_bin)
            ece += abs_diff * prop_in_bin
            mce = max(mce, abs_diff)
    return float(ece), float(mce)

def evaluate_calibration(logits_tensor, labels_tensor, num_classes=3, n_bins=10):
    probs = F.softmax(logits_tensor, dim=1).detach().cpu().numpy()
    labels = labels_tensor.detach().cpu().numpy()
    preds = np.argmax(probs, axis=1)
    
    acc = float(accuracy_score(labels, preds))
    f1 = float(f1_score(labels, preds, average='macro', zero_division=0))
    ece, mce = compute_ece_mce(probs, labels, n_bins=n_bins)
    brier = float(np.mean([brier_score_loss((labels == i).astype(int), probs[:, i]) for i in range(num_classes)]))
    nll = float(log_loss(labels, probs, labels=list(range(num_classes))))
    
    cm = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    sens = np.diag(cm) / (cm.sum(axis=1) + 1e-6)
    specs = []
    for i in range(num_classes):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp; fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specs.append(tn / (tn + fp + 1e-6))
    macro_sens = float(np.mean(sens))
    macro_spec = float(np.mean(specs))
    icbhi = float((macro_sens + macro_spec) / 2.0)
    
    return {
        'accuracy': round(acc, 4),
        'f1_macro': round(f1, 4),
        'ece': round(ece, 4),
        'mce': round(mce, 4),
        'brier_score': round(brier, 4),
        'nll': round(nll, 4),
        'icbhi_score': round(icbhi, 4),
    }


## Section 7: Base Model Training & Calibrator Fitting

In [12]:
# ============================================================
# Section 7: Base Model Training & Calibrator Fitting
# ============================================================
def train_model(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for specs, labels, _ in loader:
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(specs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(labels)
    return running_loss / max(len(loader.dataset), 1)

def extract_logits(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for specs, labels, _ in loader:
            specs = specs.to(DEVICE)
            logits = model(specs)
            all_logits.append(logits.cpu())
            all_labels.append(labels)
    return torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)

# 1. Train Base CE Model
print("[TRAIN] Training Base Model (CrossEntropyLoss) on Real Audio...")
optimizer_base = torch.optim.Adam(model_base.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
criterion_ce = nn.CrossEntropyLoss()

for epoch in range(1, CFG['num_epochs'] + 1):
    train_loss = train_model(model_base, train_loader, optimizer_base, criterion_ce)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{CFG['num_epochs']} | Train Loss: {train_loss:.4f}")

# Extract Calibration Logits
calib_logits, calib_labels = extract_logits(model_base, calib_loader)
calib_logits, calib_labels = calib_logits.to(DEVICE), calib_labels.to(DEVICE)

# 2. Fit Temperature Scaler  (was: TemperatureScaling — fixed NameError)
temp_scaler = TemperatureScaler().to(DEVICE)
opt_T = temp_scaler.fit(calib_logits, calib_labels)

# 3. Fit Vector Scaler
vector_scaler = VectorScaler(num_classes=CFG['num_classes']).to(DEVICE)
vector_scaler.fit(calib_logits, calib_labels)

# 4. Train Focal Loss Model
print("\n[TRAIN] Training Focal Loss Retrained Model on Real Audio...")
model_focal = OWMTL_DiseaseModel(num_classes=CFG['num_classes']).to(DEVICE)
optimizer_focal = torch.optim.Adam(model_focal.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
criterion_focal = FocalLoss(gamma=CFG['focal_gamma'])

for epoch in range(1, CFG['num_epochs'] + 1):
    train_loss_f = train_model(model_focal, train_loader, optimizer_focal, criterion_focal)

print("[INFO] Real audio training and calibrator fitting complete.")

[TRAIN] Training Base Model (CrossEntropyLoss) on Real Audio...
Epoch 01/30 | Train Loss: 0.8523
Epoch 10/30 | Train Loss: 0.7555
Epoch 20/30 | Train Loss: 0.5829
Epoch 30/30 | Train Loss: 0.3539
[CALIB] Temperature Scaling fitted: T = 1.8227
[CALIB] Vector Scaling fitted.

[TRAIN] Training Focal Loss Retrained Model on Real Audio...
[INFO] Real audio training and calibrator fitting complete.


## Section 8: Final Evaluation & Calibration Comparison

In [13]:
# ============================================================
# Section 8: Final Evaluation & Calibration Comparison
# ============================================================
test_logits_base, test_labels = extract_logits(model_base, test_loader)
test_logits_base = test_logits_base.to(DEVICE)
test_labels = test_labels.to(DEVICE)

res_uncalib = evaluate_calibration(test_logits_base, test_labels)
res_temp = evaluate_calibration(temp_scaler(test_logits_base), test_labels)
res_vec = evaluate_calibration(vector_scaler(test_logits_base), test_labels)

test_logits_focal, _ = extract_logits(model_focal, test_loader)
res_focal = evaluate_calibration(test_logits_focal.to(DEVICE), test_labels)

print("=" * 78)
print(f"M11 POST-HOC CALIBRATORS EVALUATION SUMMARY (REAL ICBHI TEST SET)")
print("=" * 78)
print(f"{'Variant':<22} | {'ECE (v)':<8} | {'MCE (v)':<8} | {'NLL (v)':<8} | {'Brier (v)':<8} | {'Acc (^)':<8} | {'ICBHI (^)':<8}")
print("-" * 78)
print(f"{'1. Uncalibrated':<22} | {res_uncalib['ece']:<8.4f} | {res_uncalib['mce']:<8.4f} | {res_uncalib['nll']:<8.4f} | {res_uncalib['brier_score']:<8.4f} | {res_uncalib['accuracy']:<8.4f} | {res_uncalib['icbhi_score']:<8.4f}")
print(f"{'2. Temp Scaling':<22} | {res_temp['ece']:<8.4f} | {res_temp['mce']:<8.4f} | {res_temp['nll']:<8.4f} | {res_temp['brier_score']:<8.4f} | {res_temp['accuracy']:<8.4f} | {res_temp['icbhi_score']:<8.4f}")
print(f"{'3. Vector Scaling':<22} | {res_vec['ece']:<8.4f} | {res_vec['mce']:<8.4f} | {res_vec['nll']:<8.4f} | {res_vec['brier_score']:<8.4f} | {res_vec['accuracy']:<8.4f} | {res_vec['icbhi_score']:<8.4f}")
print(f"{'4. Focal Loss':<22} | {res_focal['ece']:<8.4f} | {res_focal['mce']:<8.4f} | {res_focal['nll']:<8.4f} | {res_focal['brier_score']:<8.4f} | {res_focal['accuracy']:<8.4f} | {res_focal['icbhi_score']:<8.4f}")
print("=" * 78)

# Plot Reliability Diagram & ECE Comparison
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
methods = ['Uncalibrated', 'Temperature', 'Vector', 'Focal']
eces = [res_uncalib['ece'], res_temp['ece'], res_vec['ece'], res_focal['ece']]
sns.barplot(x=methods, y=eces, palette='viridis')
plt.title('ECE Comparison on Real Audio (Lower is Better)')
plt.ylabel('Expected Calibration Error')

plt.subplot(1, 2, 2)
mces = [res_uncalib['mce'], res_temp['mce'], res_vec['mce'], res_focal['mce']]
sns.barplot(x=methods, y=mces, palette='magma')
plt.title('MCE Comparison on Real Audio (Lower is Better)')
plt.ylabel('Maximum Calibration Error')
plt.tight_layout()
plot_path = os.path.join(CFG['results_dir'], 'ece_comparison_M11.png')
plt.savefig(plot_path)
plt.close()
print(f"✅ Calibration comparison plot saved -> {plot_path}")

# Export JSON
results_data = {
    'meta': {
        'model_id': CFG['model_id'],
        'model_name': CFG['model_name'],
        'contributor': CFG['member'],
        'date_completed': time.strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': 'Post-hoc calibration evaluated on real ICBHI audio'
    },
    'config': CFG,
    'environment': {
        'platform': PLATFORM,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0]
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'train_samples': len(df_train),
        'cal_samples': len(df_cal),
        'test_samples': len(df_test),
        'split_method': 'patient_independent_60_20_20'
    },
    'best_metrics': {
        'uncalibrated': res_uncalib,
        'temperature_scaled': res_temp,
        'vector_scaled': res_vec,
        'focal_loss': res_focal,
        'optimal_temperature': round(opt_T, 4)
    }
}

json_path = os.path.join(CFG['results_dir'], 'results_M11.json')
with open(json_path, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f"✅ Results JSON exported -> {json_path}")


M11 POST-HOC CALIBRATORS EVALUATION SUMMARY (REAL ICBHI TEST SET)
Variant                | ECE (v)  | MCE (v)  | NLL (v)  | Brier (v) | Acc (^)  | ICBHI (^)
------------------------------------------------------------------------------
1. Uncalibrated        | 0.4460   | 0.6466   | 1.8865   | 0.3176   | 0.3573   | 0.4983  
2. Temp Scaling        | 0.3444   | 0.7868   | 1.2831   | 0.2673   | 0.3573   | 0.4983  
3. Vector Scaling      | 0.0878   | 0.2071   | 0.8082   | 0.1652   | 0.6595   | 0.5083  
4. Focal Loss          | 0.1369   | 0.1706   | 1.0141   | 0.1683   | 0.6921   | 0.5129  


/tmp/ipykernel_58/1571850311.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=methods, y=eces, palette='viridis')
/tmp/ipykernel_58/1571850311.py:37: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=methods, y=mces, palette='magma')


✅ Calibration comparison plot saved -> /kaggle/working/M11_outputs/results/ece_comparison_M11.png
✅ Results JSON exported -> /kaggle/working/M11_outputs/results/results_M11.json
